# V6 — 09: Eval K-Scaling (K1–K6)

**Round 1**: K1–K4 → m=k (eval with n_episodes = chunk_size of the model)

**Round 2**: K5, K6 → m=k

| Tag | Policy              | k    | m=k  |
|-----|---------------------|------|------|
| K1  | ACT                 | 50   | 50   |
| K2  | ACT                 | 200  | 200  |
| K3  | ACM3                | 50   | 50   |
| K4  | ACM3                | 200  | 200  |
| K5  | ACM3+ICPE+SSCP      | 50   | 50   |
| K6  | ACM3+ICPE+SSCP      | 200  | 200  |

In [ ]:
import sys, os
from pathlib import Path

GPU_OFFSET = 0  # ← only line to change

sys.path.insert(0, str(Path(".").resolve()))
import common_v6 as v6

os.environ.setdefault("MUJOCO_GL", "osmesa")
os.environ.setdefault("PYOPENGL_PLATFORM", "osmesa")

TAGS_R1 = ["K1", "K2", "K3", "K4"]
TAGS_R2 = ["K5", "K6"]
TAGS_ALL = TAGS_R1 + TAGS_R2

def mk_eval_job(tag, gpu_local):
    """Use chunk_size as n_episodes for k-scaling eval."""
    _, _, chunk_size, _, _ = v6.MODEL_CONFIGS[tag]
    return (tag, gpu_local, chunk_size)

EVAL_JOBS_R1 = [mk_eval_job(tag, i) for i, tag in enumerate(TAGS_R1)]
EVAL_JOBS_R2 = [mk_eval_job(tag, i) for i, tag in enumerate(TAGS_R2)]

print("Eval jobs Round 1:", [(t, gpu, n) for t, gpu, n in EVAL_JOBS_R1])
print("Eval jobs Round 2:", [(t, gpu, n) for t, gpu, n in EVAL_JOBS_R2])

In [ ]:
print("=== Training status ===")
v6.print_training_status(TAGS_ALL)
print()
print("=== Eval status ===")
v6.print_eval_status(TAGS_ALL)

In [ ]:
print("Launching Round 1 (K1–K4, m=k) ...")
procs_r1 = v6.launch_eval(EVAL_JOBS_R1, GPU_OFFSET)
for tag in TAGS_R1:
    print(f"  tail -f {v6.get_output_dir(tag) / 'eval' / 'eval.log'}")

In [ ]:
print("Launching Round 2 (K5, K6, m=k) ...")
procs_r2 = v6.launch_eval(EVAL_JOBS_R2, GPU_OFFSET)
for tag in TAGS_R2:
    print(f"  tail -f {v6.get_output_dir(tag) / 'eval' / 'eval.log'}")

In [ ]:
# Post-processing
results = {}
for tag in TAGS_ALL:
    m = v6.build_metrics_from_eval_info(tag)
    results[tag] = m
    v6.save_metrics(tag, m)

print(f"{'TAG':<8} {'k':>5} {'LABEL':<30} {'SR':>6} {'CI_LO':>7} {'CI_HI':>7}")
print("-" * 72)
for tag in TAGS_ALL:
    _, _, chunk_size, _, _ = v6.MODEL_CONFIGS[tag]
    m = results.get(tag, {})
    sr_s = f"{m['sr']:.3f}"       if m.get('sr')       is not None else " ─"
    lo_s = f"{m['sr_ci_lo']:.3f}" if m.get('sr_ci_lo') is not None else " ─"
    hi_s = f"{m['sr_ci_hi']:.3f}" if m.get('sr_ci_hi') is not None else " ─"
    print(f"{tag:<8} {chunk_size:>5} {m.get('label', tag):<30} {sr_s:>6} {lo_s:>7} {hi_s:>7}")